# Train the TargetRanker (v2) — shallower L2, joint unfreeze, fresh init

**transformer_v2 at d_model=128 + 9-step sparse history window** `(26, 21, 16, 11, 8, 5, 2, 1, 0)`. L2 CrossEntityAttention is a 2-layer / 4-head Pre-LN Transformer. v1 ckpts (d_model=64, 3-layer L2, dense 3-step window) are shape-incompatible at every layer — the loader detects per-module mismatch and fresh-inits anything that doesn't fit. This is a full-fresh-init run; the action-encoder ckpt is no longer a useful warm-start at d=128.

**Single loss:** cross-entropy on `target_planet_idx`, supervising the TargetRanker.

**Everything below the ranker is unfrozen and trains jointly** — all 4 encoders (FleetEncoder, PlanetEncoder, PlanetEntityEncoder, CrossEntityAttention) **and** the TargetRanker (token projection + role embeddings + two MultiheadAttention blocks + scorer MLP). No PairScoreHead, no FracHead, no per-planet TargetHead — the old pair-summary path is replaced by the Stage A target→source cross-attention.

```
┌────────────── per-fleet inputs ─────────────┐   ┌──── per-planet inputs ────┐
│ fleet_features (B,T,F,FLEET_RAW_DIM)         │   │ planet_features           │
│ owner / source / target / eta / ships        │   │ planet_mask               │
└──────────────────┬───────────────────────────┘   └────────────┬──────────────┘
                   │                                            │
                   ▼                                            ▼
            ┌────────────────┐                          ┌────────────────┐
            │  FleetEncoder  │                          │  PlanetEncoder │
            └───────┬────────┘                          └───────┬────────┘
                    │                                           │
                    └─────────────────┬─────────────────────────┘
                                      │
                                      ▼
                          ┌────────────────────────────┐
                          │ PlanetEntityEncoder (L1)   │
                          │ pool fleets per (planet,   │
                          │   owner) → entity_tokens   │
                          │   (B, T, P, d)             │
                          └─────────────┬──────────────┘
                                        │
                            ┌───────────┤
                            │           ▼  (current-step skip)
                            │   entity_now (B, P, d)
                            │           │
                            │           ▼
                            │   ┌────────────────────────────┐
                            │   │ CrossEntityAttention (L2)  │
                            │   │ 2-layer Pre-LN Transformer (d=128) │
                            │   │ + CLS + step_embed         │
                            │   └───────┬──────────────┬─────┘
                            │           │              │
                            │   ctx_now (B,P,d)   glob (B,d)
                            │           │              │
                            │           │              │
                            │           │              ▼
                            ▼           ▼   target_scalars (B, P, c_agg=9)
                                                 owner_oh [fr/en/neu] (3)
                                                 garrison_log (1)
                                                 n_friendly_R / n_enemy_R (2)
                                                 nearest_enemy_dist (1)
                                                 inbound_own_h10 (1)
                                                 inbound_enemy_h10_sum (1)
                            ┌────────────────────────────────────────────────┐
                            │  Token assembly                                │
                            │   target_raw = [ctx ‖ entity_now ‖ glob ‖ tag]│
                            │   target_base = Linear(target_raw)            │
                            │   → (B, P, d_rank)                            │
                            │   (same pool serves as both src & tgt;        │
                            │    role embeddings disambiguate the two uses) │
                            └─────────────────────┬──────────────────────────┘
                                                  │
              source_keys = target_base + source_role   target_query = target_base + target_role
                                                  │
                                                  ▼
   ┌───────────────────────────────────────────────────────────────────────┐
   │ Stage A — target→source cross-attention (L3)                          │
   │                                                                       │
   │   Q = target_query   K, V = source_keys   (B, P, d_rank)              │
   │   key_padding_mask = ~src_valid      ← owned-source planets          │
   │   attn_mask = eye(P)                  ← exclude source == target      │
   │                                          (pairwise, NOT per-key)      │
   │                                                                       │
   │   source_ctx = MHA(Q, K, V, masks)                                    │
   │   source_aware = LayerNorm(target_base + source_ctx)                  │
   │                                                                       │
   │   nan_to_num guards rows where every source happens to be masked.     │
   │                                                                       │
   │   Meaning: "for each candidate target t, which owned sources          │
   │             can act on me, and how strongly?"                         │
   └─────────────────────────────────┬─────────────────────────────────────┘
                                     │
                                     ▼
   ┌───────────────────────────────────────────────────────────────────────┐
   │ Stage B — target self-attention (L4)                                  │
   │                                                                       │
   │   Q = K = V = source_aware   (B, P, d_rank)                           │
   │   key_padding_mask = ~target_valid    ← all real planets              │
   │                                                                       │
   │   rank_ctx = LayerNorm(source_aware + MHA(...))                       │
   │                                                                       │
   │   Meaning: "for each source-aware target t, how do I compare against  │
   │             every other candidate target?"                            │
   └─────────────────────────────────┬─────────────────────────────────────┘
                                     │
                                     ▼
              ┌──────────────────────────────────────────────┐
              │ Final scorer                                 │
              │  score_feat = [rank_ctx ‖ source_aware       │
              │               ‖ target_base ‖ target_scalars]│
              │   dim = 3·d_rank + c_agg                     │
              │                                              │
              │  MLP (3 layers, GELU, std=0.05 final init)   │
              │                                              │
              │  → target_logits (B, P)                      │
              └──────────────────────┬───────────────────────┘
                                     │
                                     ▼
                ┌────────────────────────────────────────┐
                │ Loss-time masking                      │
                │  target_valid = planet_mask            │
                │  force target_valid[gold_target] = T   │
                │  logits.masked_fill(~target_valid,-inf)│
                └──────────────────┬─────────────────────┘
                                   │
                                   ▼
                     CE vs target_planet_idx
                         (acted rows only)
```

`stack.unfreeze_all()` is called inside `train_target_rank_kwargs` — there is no freeze knob. Every parameter below the upstream `FleetEncoder/PlanetEncoder/PlanetEntityEncoder/CrossEntityAttention` (which are also unfrozen and updated) trains jointly under the single rank-CE loss.

**Why no PairScoreHead?** The pair head's `(B, P, P)` output was column-reduced to 5 hand-crafted scalars per target (max / top2 / lse / count / mean), throwing away per-source detail. Stage A cross-attention replaces this with a learned source-conditioned summary — for each target it directly asks "which owned sources matter, and how?". Bonus: the old pair-head's `torch.finfo.min` masking caused a subtle numerical collapse downstream (loss pinned at log(64)); the new design has no such hand-engineered masked reduce.

**Loss-time mask discipline (carried over from the bug fix in the prior iteration):** the batch's raw `tgt_valid` may be all-False when `data.tgz` ships without the `_masks/*.npz` side cache. `TargetRankerStack._build_masks` falls back to `planet_mask` for empty rows and force-includes the gold target, returning the corrected mask alongside the logits so the loss applies the same mask.

**Per-epoch metrics:** `target_loss / target_top1 / top3 / top5 / target_logit_std / uniform_ce_baseline / avg_candidate_count`. Compare loss against `uniform_ce_baseline = mean(log(n_valid_targets_per_row))`, not against the padded `log(P=64)`.

**Prerequisites in `gs://orbit-wars-shipping/`**: `code.tgz`, `data.tgz`, `weights.tgz`, **and `pair_score_assets.tgz`** (carries the action-stage ckpt that warm-starts L0/L1).

```bash
INCLUDE_PAIR_SCORE_ASSETS=1 PAIR_SCORE_PLAYER=Ebi UPLOAD=1 ./scripts/pack_for_gpu.sh
```

**Runtime**: T4 GPU strongly recommended — joint training of 4 encoders + TargetRanker is ~900k+ trainable params.

## 1. Verify GPU (or note CPU fallback)

In [ ]:
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    DEVICE = 'cuda'
else:
    print('No GPU — falling back to CPU. Joint encoders + rank head will be '
          'slow on CPU; expect ~40 min/epoch at 20k acted rows. Drop '
          'MAX_ROWS to 2000-5000 for a usable laptop loop.')
    DEVICE = 'cpu'
print(f'CUDA: {torch.version.cuda}  PyTorch: {torch.__version__}')

## 2. Configuration

**Scale-up run vs the v2-baseline (d=64, 6-step).** Two coupled changes:

- `d_model = 128` (was 64) — doubles the per-token width across L0/L1/L2.
- `HISTORY_OFFSETS = (26, 21, 16, 11, 8, 5, 2, 1, 0)` — 9-step sparse window covering recent triplet + medium anchors + long anchors.

Together these blow up `step_embed` from `(6, 64)` to `(9, 128)` and grow L0/L1 by ~4× params (linear in d_model on the encoder, quadratic on the feedforward). Full stack is ~960K trainable params (up from ~470K).

`L0/L1/L2 all fresh-init` because the available action-encoder ckpts are at d=64; the loader logs `shape/key mismatch` for each module and skips the warm-start. `unfreeze_all()` runs as usual.

- `LR = 5e-4`, `WEIGHT_DECAY = 0.0`, `DROPOUT = 0.0` — same v1-baseline schedule.
- `EPOCHS = 20` — likely want more for a full-fresh run; bump if val_top1 is still climbing.
- `D_RANK = 128`, `N_HEADS = 4` for the TargetRanker stack.

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT = 'analog-receiver-489214-e9'
BUCKET  = 'gs://orbit-wars-shipping'
PLAYER  = 'Ebi'

# === Fresh init — v2 has a 2-layer L2 so v1 ckpts are not shape-compatible. ===
INIT_FROM_GCS = None

# === Hyperparameters — matched to v1 fresh-init baseline for a clean L2-depth ablation. ===
LR           = 5e-4
WEIGHT_DECAY = 0.0
DROPOUT      = 0.0
EPOCHS       = 20
BATCH_SIZE   = 64
VAL_FRAC     = 0.2
MAX_ROWS     = None    # full Ebi acted rows
MAX_FLEETS   = 1024
D_RANK       = 128
N_HEADS      = 4

!gcloud config set project {PROJECT}

## 3. Pull tarballs (+ optional prior pair ckpt)

In [ ]:
import os
import subprocess
from pathlib import Path
WORK = '/content/orbit-wars'
os.makedirs(WORK, exist_ok=True)
%cd {WORK}

# All four tarballs are required:
#   code.tgz              — repo source
#   data.tgz              — fleet/planet/entity/cross_entity/action CSVs
#   weights.tgz           — fleet/planet/entity encoder ckpts (L0+L1)
#   pair_score_assets.tgz — action_best.pt (warm-starts L0+L1 jointly)
#                           + per-player replay subtree
def _run_gsutil(args):
    return subprocess.run(
        ['gsutil', *args],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

def _fail_pull(message):
    raise RuntimeError(message) from None

required = ('code.tgz', 'data.tgz', 'weights.tgz', 'pair_score_assets.tgz')
for name in required:
    uri = f'{BUCKET}/{name}'
    print(f'checking {uri}')
    ls = _run_gsutil(['ls', '-lh', uri])
    if ls.stdout.strip():
        print(ls.stdout.strip())
    if ls.returncode != 0:
        _fail_pull(
            f'missing required object: {uri}\n'
            f'Re-upload from your local repo with:\n'
            f'  BUCKET={BUCKET} INCLUDE_PAIR_SCORE_ASSETS=1 PAIR_SCORE_PLAYER={PLAYER} UPLOAD=1 ./scripts/pack_for_gpu.sh\n'
            f'Then verify with:\n'
            f'  gsutil ls -lh {BUCKET}/code.tgz {BUCKET}/data.tgz {BUCKET}/weights.tgz {BUCKET}/pair_score_assets.tgz'
        )
    print(f'pulling {uri}')
    cp = _run_gsutil(['cp', uri, '.'])
    if cp.stdout.strip():
        print(cp.stdout.strip())
    if cp.returncode != 0:
        _fail_pull(
            f'failed to download required object: {uri}\n'
            f'gsutil output:\n{cp.stdout}\n'
            f'Re-upload from your local repo with:\n'
            f'  BUCKET={BUCKET} INCLUDE_PAIR_SCORE_ASSETS=1 PAIR_SCORE_PLAYER={PLAYER} UPLOAD=1 ./scripts/pack_for_gpu.sh'
        )
    size = Path(name).stat().st_size if Path(name).exists() else 0
    print(f'  {name}: {size/1e6:.1f} MB')
    if size < 1024:
        _fail_pull(f'{name} downloaded but is suspiciously small ({size} bytes); re-upload it.')

## 4. Unpack + validate

In [ ]:
%cd {WORK}
!pwd
!ls -la | head

# Confirm the tarballs actually arrived in c3-pull. A failed gsutil
# (e.g. quota/auth) can leave a zero-byte placeholder.
import os
def _fail_unpack(message):
    raise RuntimeError(message) from None
for name in ('code.tgz', 'data.tgz', 'weights.tgz', 'pair_score_assets.tgz'):
    if not os.path.exists(name):
        _fail_unpack(f'{name} missing in {WORK} \u2014 c3-pull failed; re-run it.')
    sz = os.path.getsize(name)
    print(f'{name}: {sz/1e6:.1f} MB')
    if sz < 1024:
        _fail_unpack(f'{name} is suspiciously small ({sz} bytes); the gsutil download in c3-pull failed.')

# Extract.
!tar xzf code.tgz
!tar xzf data.tgz
!tar xzf weights.tgz
!tar xzf pair_score_assets.tgz   # \u2192 data/runs/action/<TS>/action_best.pt + data/replays/<PLAYER>/

# Show what landed under data/datasets/* so a 0-count branch is debuggable.
import glob
print()
print('---- post-extraction inventory under data/datasets/ ----')
for sub in ('action', 'planet', 'fleet', 'entity', 'cross_entity'):
    n = len(glob.glob(f'data/datasets/{sub}/*.csv'))
    print(f'  data/datasets/{sub}: {n} CSVs')
print()

from pathlib import Path

required = (
    ('data/datasets/action', 'action_*.csv', 'action CSVs'),
    ('data/datasets/planet', 'planet_*.csv', 'planet CSVs'),
    ('data/datasets/fleet', 'fleet_*.csv', 'fleet CSVs'),
    ('data/datasets/entity', 'entity_*.csv', 'entity CSVs'),
    ('data/datasets/cross_entity', 'cross_entity_*.csv', 'cross-entity CSVs'),
)
for rel, pattern, label in required:
    count = len(list(Path(rel).glob(pattern)))
    if count == 0:
        d = Path(rel)
        if not d.exists():
            print(f'  ERROR: {rel} does not exist post-extraction.')
            print(f'         tar xzf data.tgz may have failed silently.')
            print(f'         Run: !tar tzf data.tgz | head 20  in a separate cell')
            print(f'         to confirm the tarball still has the expected paths.')
        else:
            print(f'  ERROR: {rel} exists but contains nothing matching {pattern!r}.')
            print(f'         Contents: {list(d.iterdir())[:10]}')
        _fail_unpack(f'no {label} found under {rel}.')
    print(f'{label}: {count}')

# Encoder ckpt discovery. action_best.pt comes from pair_score_assets.tgz
# and lives at data/runs/action/<TS>/action_best.pt.
ENCODER_CKPT_OVERRIDE = None    # set to a path string to skip discovery
ENCODER_CKPT = None
if ENCODER_CKPT_OVERRIDE:
    if Path(ENCODER_CKPT_OVERRIDE).exists():
        ENCODER_CKPT = ENCODER_CKPT_OVERRIDE
    else:
        _fail_unpack(f'ENCODER_CKPT_OVERRIDE points to a missing file: {ENCODER_CKPT_OVERRIDE}')
else:
    for pat in (
        'data/runs/action/*/action_best.pt',
        'data/runs/**/action_best.pt',
        'data/runs/**/*action_best*.pt',
    ):
        hits = sorted(glob.glob(pat, recursive=True))
        if hits:
            ENCODER_CKPT = hits[-1]
            break
if not ENCODER_CKPT:
    print('no action_best.pt found. Layout under data/runs/:')
    for p in sorted(Path('data/runs').rglob('*.pt'))[:20]:
        print(f'  {p}')
    _fail_unpack(
        'pair_score_assets.tgz did not yield an action ckpt. Re-pack with '
        'INCLUDE_PAIR_SCORE_ASSETS=1 and PAIR_SCORE_PLAYER set, and verify '
        'a data/runs/action/<TS>/action_best.pt exists locally before packing.'
    )
print('encoder ckpt:', ENCODER_CKPT)

player_replays = sorted(glob.glob(f'data/replays/{PLAYER}/*.json.gz'))
if not player_replays:
    _fail_unpack(f'no replays under data/replays/{PLAYER}/.')
print(f'replays for {PLAYER}: {len(player_replays)}')

## 5. Install + import

In [ ]:
%cd {WORK}
!pip install -q -r requirements.txt --no-deps
!pip install -q kaggle-environments

In [ ]:
import sys
sys.path.insert(0, WORK)

from agents.transformer_v2.pretrain.pair_score import prepare_dataset
from agents.transformer_v2.pretrain.target_rank import train_target_rank_kwargs
print('imports OK — target_rank training ready')

### 5b. Materialize the dataset (run once per session)

Parses Ebi's CSVs into snapshot tensors and holds them in the kernel variable `dataset`. Cache is disabled by default — if you want persistence across kernel restarts, pass `cache_dir='data/datasets/_cache'` (warning: ~9 GB for full Ebi at max_fleets=1024, n_history=9 sparse [26,21,16,11,8,5,2,1,0]).

In [ ]:
dataset = prepare_dataset(
    player=PLAYER,
    filter_mode='all',
    max_planets=64,
    max_fleets=MAX_FLEETS,
    n_history=9,
    cache_dir=None,
    rebuild_cache=False,
)
print(f'dataset ready: {len(dataset)} snapshots held in kernel')

## 6. Train the TargetRanker (v2 / d=128 / T=9)

`train_target_rank_kwargs` with `init_from=None`. The loader will print `shape/key mismatch` for L0/L1/L2 because the action-encoder ckpt is at d_model=64 — that is expected for the d=128 scale-up. `unfreeze_all()` still runs.

**Per-epoch line:**

```
[target_rank] ep=  N tr_loss=...  tr_top1=...  tr_std=...  |  val_loss=...  val_top1=...  ...  baseline_ce=...  avg_cand=...
```

**Expected trajectory (full-fresh, d=128, 9-step):**
- Epoch 1: val_top1 ≈ 0.08–0.12 — slower start than warm-started runs.
- Epoch 5–10: val_top1 climbing past 0.25.
- Late epochs (15–20): target val_top1 ≥ 0.45 if the extra capacity + longer window is worth it. If still below 0.40 by epoch 20, the scale change is hurting more than helping — fall back to d=64 / 6-step.

Expect ~2× per-epoch wall time vs the d=64 run (more params + 50% more history slots in attention).

In [ ]:
import time
TS = time.strftime('%Y%m%d-%H%M%S')
OUT_DIR = f'data/runs/target_rank_v2/{PLAYER}_{TS}'
print('out dir:', OUT_DIR)

# Pull the prior ckpt locally if we're resuming from one.
INIT_FROM_LOCAL = None
if INIT_FROM_GCS:
    INIT_FROM_LOCAL = INIT_FROM_GCS.replace(f'{BUCKET}/', '')
    import os
    os.makedirs(os.path.dirname(INIT_FROM_LOCAL), exist_ok=True)
    !gsutil cp {INIT_FROM_GCS} {INIT_FROM_LOCAL}
    print('init seed:', INIT_FROM_LOCAL)

best_ckpt = train_target_rank_kwargs(
    encoder_ckpt=ENCODER_CKPT,
    out_dir=OUT_DIR,
    player=PLAYER,
    filter='all',
    max_rows=MAX_ROWS,
    val_frac=VAL_FRAC,
    batch_size=BATCH_SIZE,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    epochs=EPOCHS,
    max_fleets=MAX_FLEETS,
    d_rank=D_RANK,
    n_heads=N_HEADS,
    dropout=DROPOUT,
    device=DEVICE,
    dataset=dataset,
    init_from=INIT_FROM_LOCAL,
)
print('done. best ckpt:', best_ckpt)

import torch
ck = torch.load(best_ckpt, map_location='cpu', weights_only=False)
print('ckpt keys:', sorted(k for k in ck if not k.startswith('_')))

## 7. Log summary

In [ ]:
import json
log = json.loads(open(f'{WORK}/{OUT_DIR}/log.json').read())
for e in log:
    tr, v = e['train'], e['val']
    print(
        f"ep {e['epoch']:2d}  "
        f"tr_loss={tr.get('target_loss', 0):.3f}  "
        f"tr_top1={tr.get('target_top1', 0):.3f}  "
        f"tr_std={tr.get('target_logit_std', 0):.3f}  |  "
        f"val_loss={v.get('target_loss', 0):.3f}  "
        f"val_top1={v.get('target_top1', 0):.3f}  "
        f"val_top3={v.get('target_top3', 0):.3f}  "
        f"val_top5={v.get('target_top5', 0):.3f}  "
        f"baseline={v.get('uniform_ce_baseline', 0):.3f}  "
        f"cand={v.get('avg_candidate_count', 0):.1f}"
    )
best = max(log, key=lambda e: e['val'].get('target_top1', 0))
print(f"\nbest val_top1={best['val'].get('target_top1', 0):.3f} (epoch {best['epoch']})")

## 8. Push results to GCS

In [ ]:
%cd {WORK}
!gsutil -m cp -r {OUT_DIR} {BUCKET}/runs/
print(f'uploaded {BUCKET}/runs/{OUT_DIR.rsplit("/", 1)[-1]}')